# Logistic Workflow Notebook

This notebook models the distribution of cumulative returns through that week's Friday, standardizes the target by horizon, and derives expected realised volatility from the predicted second moment.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    raise FileNotFoundError("Run this notebook from the project root so `src/` is available.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import DATA_OUTPUT_DIR
from logistic import (
    build_return_features,
    add_weekly_friday_target,
    add_quantile_targets,
    fit_balanced_bucket_models,
    compute_expected_realised_volatility,
)


In [ ]:
feature_candidates = [
    DATA_OUTPUT_DIR / "feature_table_stage1_stage2_notebook.csv",
    DATA_OUTPUT_DIR / "feature_table.csv",
]

feature_path = next((path for path in feature_candidates if path.exists()), None)
if feature_path is None:
    raise FileNotFoundError(
        "Could not find a feature table. Checked: "
        + ", ".join(str(path) for path in feature_candidates)
    )

feature_df = pd.read_csv(feature_path, parse_dates=["date"])
print(f"Using feature table: {feature_path}")
feature_df[["date", "log_return", "atm_iv_30d"]].head()

## Step 1: Build lagged returns and calendar features

In [ ]:
feature_engineered_df = build_return_features(feature_df, return_col="log_return")
feature_engineered_df[["date", "weekday", "days_to_friday", "log_return", "rolling_std_5", "rolling_std_22"]].head(10)

## Step 2: Build the Friday-expiry target and horizon-standardized target

In [ ]:
target_ready_df = add_weekly_friday_target(feature_engineered_df, return_col="log_return")
target_ready_df[["date", "weekday", "days_to_friday", "target_forward_return", "target_horizon_scaled_return"]].head(10)

## Step 3: Create 5 quantile categories from the standardized target

In [ ]:
N_QUANTILES = 5
target_df, quantile_info = add_quantile_targets(
    target_ready_df,
    target_col="target_horizon_scaled_return",
    n_quantiles=N_QUANTILES,
)
quantile_info

In [ ]:
dummy_cols = [col for col in target_df.columns if col.endswith("_dummy")]
target_df[["date", "days_to_friday", "target_forward_return", "target_horizon_scaled_return", "quantile_category", *dummy_cols]].head(10)

## Step 4: Fit one balanced logistic regression per quantile

In [ ]:
feature_cols = [
    c
    for c in target_df.columns
    if c.startswith("log_return_lag_")
    or c.startswith("abs_log_return_lag_")
    or c.startswith("rolling_")
    or c in {"weekday", "days_to_friday"}
]

modeled_df, models = fit_balanced_bucket_models(
    target_df,
    feature_cols=feature_cols,
    quantile_info=quantile_info,
)
print(f"Number of fitted models: {len(models)}")

In [ ]:
prob_cols = [col for col in modeled_df.columns if col.endswith("_prob")]
modeled_df[["date", "days_to_friday", "target_horizon_scaled_return", "quantile_category", *prob_cols]].dropna().head(10)

## Step 5: Derive expected realised volatility from the standardized return distribution

In [ ]:
result_df = compute_expected_realised_volatility(modeled_df, quantile_info)
result_df[[
    "date",
    "atm_iv_30d",
    "days_to_friday",
    "target_forward_return",
    "target_horizon_scaled_return",
    "expected_scaled_return_sq_plain",
    "expected_realised_volatility_plain",
    "expected_realised_volatility",
]].dropna().head(10)

In [ ]:
result_df[[
    "atm_iv_30d",
    "expected_realised_volatility_plain",
    "expected_realised_volatility",
]].dropna().describe()

## Step 6: Save outputs

In [ ]:
output_path = DATA_OUTPUT_DIR / "logistic_workflow_expected_rvol.csv"
quantile_output_path = DATA_OUTPUT_DIR / "logistic_workflow_quantiles.csv"

result_df.to_csv(output_path, index=False)
quantile_info.to_csv(quantile_output_path, index=False)

print(f"Saved modeled dataframe to {output_path}")
print(f"Saved quantile metadata to {quantile_output_path}")